# Football Tracking Data — Preprocessing & Detection Analysis

End-to-end analysis notebook covering:
1. Data loading & quality audit
2. Detection quality (confidence, class balance, per-frame density)
3. Tracking stability (track lengths, ID coverage)
4. Team segmentation results
5. Ball detection & interpolation
6. Possession analysis
7. Player kinematics (speed distributions, top movers)
8. Spatial heatmaps
9. Summary dashboard

In [ ]:
# Install analysis dependencies (run once if not already installed)
# pandas and seaborn are not included in the default tracking env
import importlib, subprocess, sys

required = {
    'pandas':     'pandas>=2.0',
    'matplotlib': 'matplotlib>=3.7',
    'seaborn':    'seaborn>=0.13',
    'scipy':      'scipy>=1.10',
}
for module, pkg in required.items():
    if importlib.util.find_spec(module) is None:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All dependencies present.')

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

BASE_DIR  = Path('.')
ANNOT_DIR = BASE_DIR / 'data' / 'annotations'

def find_csv(stem):
    for name in [f'{stem}.csv', f'{stem} (2).csv', f'{stem} (1).csv']:
        p = ANNOT_DIR / name
        if p.exists():
            return p
    raise FileNotFoundError(f'Cannot find {stem}.csv in {ANNOT_DIR}')

TRACKING_CSV   = find_csv('tracking')
ENRICHED_CSV   = find_csv('tracking_enriched')
METADATA_CSV   = find_csv('metadata')
PLAYER_SUM_CSV = find_csv('player_summary')
POSS_SUM_CSV   = find_csv('possession_summary')

print('Data files found:')
for p in [TRACKING_CSV, ENRICHED_CSV, METADATA_CSV, PLAYER_SUM_CSV, POSS_SUM_CSV]:
    print(f'  {p.name:<45} {p.stat().st_size/1024:>8.1f} KB')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
TEAM_COLORS  = {0: '#2196F3', 1: '#F44336', -1: '#9E9E9E'}
TEAM_LABELS  = {0: 'Team A', 1: 'Team B', -1: 'Unknown'}
CLASS_LABELS = {0: 'Ball', 1: 'Goalkeeper', 2: 'Player', 3: 'Referee'}
CLASS_COLORS = {0: '#FFD600', 1: '#4CAF50', 2: '#2196F3', 3: '#FF9800'}
print('Setup complete.')


---
## 1 · Data Loading & Preprocessing Quality Audit

In [ ]:
tracking   = pd.read_csv(TRACKING_CSV)
enriched   = pd.read_csv(ENRICHED_CSV)
metadata   = pd.read_csv(METADATA_CSV)
player_sum = pd.read_csv(PLAYER_SUM_CSV)
poss_sum   = pd.read_csv(POSS_SUM_CSV)

print('=== TRACKING (raw) ===')
print(f'  Rows: {len(tracking):,}   Cols: {tracking.shape[1]}')
print(tracking.dtypes.to_string())
print('\n=== ENRICHED (with kinematics) ===')
print(f'  Rows: {len(enriched):,}   Cols: {enriched.shape[1]}')
print(enriched.dtypes.to_string())


In [ ]:
def missing_report(df, name):
    mv = df.isnull().sum()
    mv = mv[mv > 0]
    if mv.empty:
        print(f'{name}: no missing values')
        return
    print(f'\n{name} — missing values:')
    for col, cnt in mv.items():
        print(f'  {col:<25} {cnt:>7,}  ({100*cnt/len(df):.1f}%)')

missing_report(tracking,   'TRACKING')
missing_report(enriched,   'ENRICHED')
missing_report(metadata,   'METADATA')
missing_report(player_sum, 'PLAYER_SUMMARY')


In [ ]:
n_frames  = tracking['frame_id'].nunique()
n_objects = tracking['object_id'].nunique()
conf_mean = tracking['confidence'].mean()
conf_low  = (tracking['confidence'] < 0.5).sum()

print(f'Frames with detections : {n_frames:,}')
print(f'Unique tracked objects : {n_objects}')
print(f'Class IDs seen         : {sorted(tracking["class_id"].unique())}')
print(f'Mean confidence        : {conf_mean:.3f}')
print(f'Low-confidence (<0.5)  : {conf_low:,}  ({100*conf_low/len(tracking):.1f}%)')

cls_counts = tracking['class_id'].map(CLASS_LABELS).value_counts()
print('\nDetection count by class:')
for cls, cnt in cls_counts.items():
    print(f'  {cls:<15} {cnt:>8,}')


In [ ]:
# ── Preprocessing: build clean enriched df ───────────────────────────────────
df = enriched.copy()

for col in ['ball_x', 'ball_y']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['class_label'] = df['class_id'].map(CLASS_LABELS).fillna('Unknown')
df['team_label']  = df['team_id'].map(TEAM_LABELS).fillna('Unknown')
df['bbox_w']      = df['bbox_x2'] - df['bbox_x1']
df['bbox_h']      = df['bbox_y2'] - df['bbox_y1']
df['bbox_area']   = df['bbox_w'] * df['bbox_h']
df['cx']          = (df['bbox_x1'] + df['bbox_x2']) / 2
df['cy']          = (df['bbox_y1'] + df['bbox_y2']) / 2

SPEED_CAP = 100.0
outlier_speed = (df['speed_km_h'] > SPEED_CAP).sum()
df['speed_km_h_clean'] = df['speed_km_h'].clip(upper=SPEED_CAP)

frame_max = int(df['frame_id'].max())

print(f'Speed outliers capped at {SPEED_CAP} km/h: {outlier_speed:,} rows')
print('Preprocessed dataframe ready:', df.shape)
df.head(3)


---
## 2 · Detection Quality

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Detection Quality Dashboard', fontsize=15, fontweight='bold')

# Confidence distribution
ax = axes[0, 0]
for cls_id, grp in tracking.groupby('class_id'):
    ax.hist(grp['confidence'], bins=40, alpha=0.65,
            label=CLASS_LABELS.get(cls_id, str(cls_id)),
            color=CLASS_COLORS.get(cls_id, 'grey'), density=True)
ax.axvline(0.5, color='red', lw=1.5, ls='--', label='threshold=0.5')
ax.set_title('Confidence Distribution by Class')
ax.set_xlabel('Confidence'); ax.set_ylabel('Density'); ax.legend(fontsize=8)

# Detections per frame
ax = axes[0, 1]
dets_per_frame = tracking.groupby('frame_id').size()
ax.plot(dets_per_frame.index, dets_per_frame.values, lw=0.8, alpha=0.7, color='steelblue')
ax.fill_between(dets_per_frame.index, dets_per_frame.values, alpha=0.2, color='steelblue')
ax.axhline(dets_per_frame.mean(), color='red', ls='--', lw=1.5, label=f'mean={dets_per_frame.mean():.1f}')
ax.set_title('Detections per Frame'); ax.set_xlabel('Frame'); ax.set_ylabel('Count'); ax.legend()

# Class pie
ax = axes[0, 2]
cls_cnt = tracking['class_id'].value_counts()
labels  = [CLASS_LABELS.get(i, str(i)) for i in cls_cnt.index]
colors  = [CLASS_COLORS.get(i, 'grey') for i in cls_cnt.index]
wedges, texts, autotexts = ax.pie(cls_cnt.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90, pctdistance=0.82)
for at in autotexts: at.set_fontsize(9)
ax.set_title('Class Distribution')

# BBox area distribution
ax = axes[1, 0]
for cls_id in [2, 1, 0]:
    sub = df[df['class_id'] == cls_id]
    ax.hist(sub['bbox_area'].clip(upper=8000), bins=50, alpha=0.65,
            label=CLASS_LABELS.get(cls_id), color=CLASS_COLORS.get(cls_id, 'grey'), density=True)
ax.set_title('Bounding-Box Area Distribution')
ax.set_xlabel('BBox Area (px²)'); ax.set_ylabel('Density'); ax.legend(fontsize=8)

# Confidence rolling avg over time
ax = axes[1, 1]
conf_by_frame = tracking.groupby('frame_id')['confidence'].mean()
rolling = conf_by_frame.rolling(30, min_periods=1).mean()
ax.plot(conf_by_frame.index, conf_by_frame.values, lw=0.5, alpha=0.4, color='steelblue')
ax.plot(rolling.index, rolling.values, lw=2, color='steelblue', label='30-frame rolling')
ax.axhline(0.5, color='red', ls='--', lw=1.2, label='threshold')
ax.set_ylim(0, 1); ax.set_title('Mean Confidence Over Time')
ax.set_xlabel('Frame'); ax.set_ylabel('Confidence'); ax.legend(fontsize=8)

# Confidence boxplots by class
ax = axes[1, 2]
plot_df = tracking.copy()
plot_df['class_label'] = plot_df['class_id'].map(CLASS_LABELS)
present_ids = sorted(plot_df['class_id'].unique())
order  = [CLASS_LABELS[k] for k in present_ids if k in CLASS_LABELS]
colors2 = [CLASS_COLORS.get(k, 'grey') for k in present_ids if k in CLASS_LABELS]
sns.boxplot(data=plot_df, x='class_label', y='confidence', order=order,
            palette=colors2, ax=ax, width=0.5)
ax.axhline(0.5, color='red', ls='--', lw=1.2)
ax.set_title('Confidence per Class'); ax.set_xlabel(''); ax.set_ylabel('Confidence')

plt.tight_layout()
plt.savefig('data/annotations/detection_quality.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/annotations/detection_quality.png')


---
## 3 · Tracking Stability

In [ ]:
track_lengths = tracking.groupby('object_id').size().rename('track_len')
track_meta    = tracking.groupby('object_id').first()[['class_id', 'team_id']]
tracks        = track_meta.join(track_lengths)
tracks['class_label'] = tracks['class_id'].map(CLASS_LABELS).fillna('Unknown')

players_df = tracks[tracks['class_id'].isin([1, 2])]
print('Track-length stats (players + goalkeepers):')
print(players_df['track_len'].describe().to_string())
print(f'\nShort tracks (<10 frames) : {(players_df["track_len"] < 10).sum()}')
print(f'Long  tracks (>100 frames): {(players_df["track_len"] > 100).sum()}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Tracking Stability', fontsize=14, fontweight='bold')

# Track-length histogram
ax = axes[0]
ax.hist(players_df['track_len'], bins=40, color='steelblue', edgecolor='white', lw=0.5)
ax.axvline(players_df['track_len'].median(), color='red', ls='--', lw=1.5,
           label=f'median={players_df["track_len"].median():.0f}')
ax.set_title('Track-Length Distribution\n(players)'); ax.set_xlabel('Frames'); ax.set_ylabel('Count'); ax.legend()

# Frame coverage heatstrip (top-30 longest)
ax = axes[1]
top30 = players_df.nlargest(30, 'track_len').index.tolist()
coverage_mat = np.zeros((len(top30), frame_max + 1), dtype=np.uint8)
for row_idx, obj_id in enumerate(top30):
    frames_seen = tracking[tracking['object_id'] == obj_id]['frame_id'].values
    coverage_mat[row_idx, frames_seen] = 1
ax.imshow(coverage_mat, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_title('Frame Coverage\n(top-30 tracks)'); ax.set_xlabel('Frame'); ax.set_ylabel('Track rank')
ax.set_yticks(range(len(top30)))
ax.set_yticklabels([f'ID {i}' for i in top30], fontsize=6)

# Detection gap distribution
ax = axes[2]
gap_sizes = []
for obj_id, grp in tracking.groupby('object_id'):
    frames = np.sort(grp['frame_id'].values)
    if len(frames) > 1:
        gaps = np.diff(frames)
        gap_sizes.extend(gaps[gaps > 1].tolist())
if gap_sizes:
    ax.hist(gap_sizes, bins=50, color='salmon', edgecolor='white', lw=0.5)
    ax.set_title(f'Detection Gaps (n={len(gap_sizes):,})')
    ax.set_xlabel('Gap (frames)'); ax.set_ylabel('Count'); ax.set_xlim(1)
else:
    ax.text(0.5, 0.5, 'No gaps', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('data/annotations/tracking_stability.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 4 · Team Segmentation

In [ ]:
team_df = df[df['class_id'].isin([1, 2])].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Team Segmentation', fontsize=14, fontweight='bold')

# Team assignment counts
ax = axes[0]
team_counts = team_df['team_id'].value_counts().sort_index()
bars = ax.bar([TEAM_LABELS.get(t, str(t)) for t in team_counts.index],
              team_counts.values,
              color=[TEAM_COLORS.get(t, 'grey') for t in team_counts.index],
              edgecolor='white')
for bar, val in zip(bars, team_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200, f'{val:,}', ha='center', fontsize=10)
ax.set_title('Detection Rows per Team'); ax.set_ylabel('Row count')

# Team detections over time
ax = axes[1]
bin_size = max(1, frame_max // 60)
team_df = team_df.copy()
team_df['frame_bin'] = (team_df['frame_id'] // bin_size) * bin_size
for team_id, color in TEAM_COLORS.items():
    sub = team_df[team_df['team_id'] == team_id]
    if sub.empty: continue
    cnt = sub.groupby('frame_bin').size()
    ax.plot(cnt.index, cnt.values, lw=1.5, color=color, label=TEAM_LABELS.get(team_id, str(team_id)), alpha=0.85)
ax.set_title('Team Detections Over Time'); ax.set_xlabel('Frame'); ax.set_ylabel('Detections (binned)'); ax.legend()

# Unique players per team
ax = axes[2]
players_per_team = player_sum[player_sum['class_id'].isin([1, 2])].groupby('team_id')['object_id'].count()
bars = ax.bar([TEAM_LABELS.get(t, str(t)) for t in players_per_team.index],
              players_per_team.values,
              color=[TEAM_COLORS.get(t, 'grey') for t in players_per_team.index], edgecolor='white')
for bar, val in zip(bars, players_per_team.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, str(val), ha='center', fontsize=11)
ax.set_title('Unique Tracked Players per Team'); ax.set_ylabel('Player count')

plt.tight_layout()
plt.savefig('data/annotations/team_segmentation.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 5 · Ball Detection & Interpolation

In [ ]:
ball_rows = df[df['ball_x'].notna()].drop_duplicates('frame_id').sort_values('frame_id')
total_frames_with_ball = len(ball_rows)
interp_frames = ball_rows['ball_interpolated'].sum() if 'ball_interpolated' in ball_rows else 0
direct_frames = total_frames_with_ball - interp_frames

print(f'Frames with ball info   : {total_frames_with_ball:,}')
print(f'  Direct detections     : {direct_frames:,}  ({100*direct_frames/max(1,total_frames_with_ball):.1f}%)')
print(f'  Interpolated          : {int(interp_frames):,}  ({100*interp_frames/max(1,total_frames_with_ball):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ball Detection & Interpolation', fontsize=14, fontweight='bold')

ax = axes[0]
if 'ball_interpolated' in ball_rows.columns:
    direct = ball_rows[ball_rows['ball_interpolated'] == 0]
    interp = ball_rows[ball_rows['ball_interpolated'] == 1]
    ax.scatter(direct['ball_x'], direct['ball_y'], s=6, alpha=0.5, color='#FFD600', label='Detected', zorder=3)
    ax.scatter(interp['ball_x'], interp['ball_y'], s=6, alpha=0.5, color='#FF5722', marker='x', label='Interpolated', zorder=3)
else:
    ax.scatter(ball_rows['ball_x'], ball_rows['ball_y'], s=4, alpha=0.5, color='#FFD600')
ax.set_title('Ball Trajectory'); ax.set_xlabel('X (px)'); ax.set_ylabel('Y (px)')
ax.invert_yaxis(); ax.legend(markerscale=2)

ax = axes[1]
ax.pie([direct_frames, int(interp_frames)],
       labels=['Direct', 'Interpolated'], colors=['#FFD600', '#FF5722'],
       autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Ball: Direct vs Interpolated')

plt.tight_layout()
plt.savefig('data/annotations/ball_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 6 · Possession Analysis

In [ ]:
print('=== Possession Summary ===')
print(poss_sum.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Possession Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
poss_sorted = poss_sum.sort_values('team_id')
labels = [TEAM_LABELS.get(t, str(t)) for t in poss_sorted['team_id']]
colors = [TEAM_COLORS.get(t, 'grey')  for t in poss_sorted['team_id']]
wedges, texts, autotexts = ax.pie(
    poss_sorted['possession_pct'], labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}, pctdistance=0.75)
ax.add_artist(plt.Circle((0, 0), 0.55, fc='white'))
ax.set_title('Team Possession %')
for at in autotexts: at.set_fontsize(12); at.set_fontweight('bold')

ax = axes[1]
poss_timeline = df[df['has_possession'] == 1][['frame_id', 'team_id']].drop_duplicates('frame_id')
if not poss_timeline.empty:
    bin_size = max(1, frame_max // 80)
    poss_timeline = poss_timeline.copy()
    poss_timeline['bin'] = (poss_timeline['frame_id'] // bin_size) * bin_size
    for team_id, color in [(0, TEAM_COLORS[0]), (1, TEAM_COLORS[1])]:
        sub = poss_timeline[poss_timeline['team_id'] == team_id]
        if sub.empty: continue
        cnt = sub.groupby('bin').size()
        ax.bar(cnt.index, cnt.values, width=bin_size * 0.9, color=color, alpha=0.75, label=TEAM_LABELS[team_id])
    ax.set_title('Possession Over Time'); ax.set_xlabel('Frame')
    ax.set_ylabel('Possession events (binned)'); ax.legend()
else:
    ax.text(0.5, 0.5, 'No possession data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('data/annotations/possession_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 7 · Player Kinematics

In [ ]:
kin_df = df[(df['class_id'].isin([1, 2])) & (df['speed_km_h_clean'] > 0)].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Player Kinematics', fontsize=14, fontweight='bold')

# Speed distribution by team
ax = axes[0, 0]
for team_id in [0, 1, -1]:
    sub = kin_df[kin_df['team_id'] == team_id]['speed_km_h_clean']
    if sub.empty: continue
    ax.hist(sub, bins=60, alpha=0.6, density=True, color=TEAM_COLORS[team_id], label=TEAM_LABELS[team_id])
ax.set_title('Speed Distribution by Team'); ax.set_xlabel('Speed (km/h)'); ax.set_ylabel('Density'); ax.legend()

# Top-10 fastest players
ax = axes[0, 1]
top_speed = player_sum[player_sum['class_id'].isin([1, 2])].nlargest(10, 'top_speed_km_h').sort_values('top_speed_km_h')
ax.barh(range(len(top_speed)), top_speed['top_speed_km_h'],
        color=[TEAM_COLORS.get(t, 'grey') for t in top_speed['team_id']], edgecolor='white')
ax.set_yticks(range(len(top_speed)))
ax.set_yticklabels([f'ID {i}' for i in top_speed['object_id']])
ax.set_title('Top-10 Peak Speeds'); ax.set_xlabel('Top speed (km/h)')
patches = [mpatches.Patch(color=TEAM_COLORS[t], label=TEAM_LABELS[t]) for t in [0, 1, -1] if t in top_speed['team_id'].values]
ax.legend(handles=patches, fontsize=8)

# Avg vs top speed scatter
ax = axes[1, 0]
ps = player_sum[player_sum['class_id'].isin([1, 2])]
for team_id in [0, 1, -1]:
    sub = ps[ps['team_id'] == team_id]
    if sub.empty: continue
    ax.scatter(sub['avg_speed_km_h'], sub['top_speed_km_h'], c=TEAM_COLORS[team_id],
               label=TEAM_LABELS[team_id], s=60, alpha=0.8, edgecolors='white', linewidths=0.5)
ax.set_title('Avg Speed vs Top Speed'); ax.set_xlabel('Avg speed (km/h)'); ax.set_ylabel('Top speed (km/h)'); ax.legend()

# Speed over time (rolling team avg)
ax = axes[1, 1]
for team_id in [0, 1]:
    sub = kin_df[kin_df['team_id'] == team_id]
    if sub.empty: continue
    spd_by_frame = sub.groupby('frame_id')['speed_km_h_clean'].mean()
    rolled = spd_by_frame.rolling(30, min_periods=1).mean()
    ax.plot(rolled.index, rolled.values, lw=2, color=TEAM_COLORS[team_id], label=TEAM_LABELS[team_id], alpha=0.9)
ax.set_title('Avg Team Speed Over Time (30-frame rolling)')
ax.set_xlabel('Frame'); ax.set_ylabel('Speed (km/h)'); ax.legend()

plt.tight_layout()
plt.savefig('data/annotations/kinematics.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 8 · Spatial Heatmaps

In [ ]:
FRAME_W = int(df['bbox_x2'].max() * 1.05)
FRAME_H = int(df['bbox_y2'].max() * 1.05)

def team_heatmap(ax, team_id, title, cmap):
    sub = df[(df['team_id'] == team_id) & (df['class_id'].isin([1, 2]))]
    if sub.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        return
    heatmap, _, _ = np.histogram2d(sub['cx'].values, sub['cy'].values,
        bins=60, range=[[0, FRAME_W], [0, FRAME_H]])
    ax.imshow(heatmap.T, origin='upper', aspect='auto',
              extent=[0, FRAME_W, FRAME_H, 0], cmap=cmap, interpolation='gaussian')
    ax.set_title(title); ax.set_xlabel('X (px)'); ax.set_ylabel('Y (px)')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Player Position Heatmaps', fontsize=14, fontweight='bold')

team_heatmap(axes[0],  0, 'Team A',       'Blues')
team_heatmap(axes[1],  1, 'Team B',       'Reds')
team_heatmap(axes[2], -1, 'Unknown/Refs', 'Greys')

plt.tight_layout()
plt.savefig('data/annotations/heatmaps.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 9 · Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 11))
fig.suptitle('Football Tracking — Summary Dashboard', fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# KPI cards
kpis = [
    ('Total frames\nwith detections', f'{n_frames:,}'),
    ('Tracked objects', str(n_objects)),
    ('Mean confidence', f'{conf_mean:.2f}'),
    ('Ball interpolation', f'{100*interp_frames/max(1,total_frames_with_ball):.1f}%'),
]
for col, (label, value) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, col])
    ax.set_facecolor('#F5F5F5')
    ax.text(0.5, 0.62, value, ha='center', va='center', transform=ax.transAxes,
            fontsize=22, fontweight='bold', color='#1A237E')
    ax.text(0.5, 0.25, label, ha='center', va='center', transform=ax.transAxes,
            fontsize=10, color='#555')
    ax.axis('off')

# Detections per frame
ax = fig.add_subplot(gs[1, 0:2])
dets_per_frame = tracking.groupby('frame_id').size()
ax.fill_between(dets_per_frame.index, dets_per_frame.values, alpha=0.3, color='steelblue')
ax.plot(dets_per_frame.index, dets_per_frame.rolling(30, min_periods=1).mean(),
        color='steelblue', lw=2)
ax.set_title('Detections / Frame (30-frame rolling)', fontsize=11)
ax.set_xlabel('Frame'); ax.set_ylabel('Count')

# Possession donut
ax = fig.add_subplot(gs[1, 2])
poss_sorted = poss_sum.sort_values('team_id')
ax.pie(poss_sorted['possession_pct'],
       labels=[TEAM_LABELS.get(t, str(t)) for t in poss_sorted['team_id']],
       colors=[TEAM_COLORS.get(t, 'grey') for t in poss_sorted['team_id']],
       autopct='%1.0f%%', startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2}, pctdistance=0.7)
ax.add_artist(plt.Circle((0, 0), 0.55, fc='white'))
ax.set_title('Possession', fontsize=11)

# Top-5 speeds
ax = fig.add_subplot(gs[1, 3])
top5 = player_sum[player_sum['class_id'].isin([1, 2])].nlargest(5, 'top_speed_km_h').sort_values('top_speed_km_h')
ax.barh(range(len(top5)), top5['top_speed_km_h'],
        color=[TEAM_COLORS.get(t, 'grey') for t in top5['team_id']], edgecolor='white')
ax.set_yticks(range(len(top5)))
ax.set_yticklabels([f'ID {i}' for i in top5['object_id']], fontsize=9)
ax.set_title('Top-5 Peak Speeds', fontsize=11); ax.set_xlabel('km/h')

# All-player heatmap
ax = fig.add_subplot(gs[2, 0:2])
all_players = df[df['class_id'].isin([1, 2])]
hm, _, _ = np.histogram2d(all_players['cx'].values, all_players['cy'].values,
    bins=80, range=[[0, FRAME_W], [0, FRAME_H]])
ax.imshow(hm.T, origin='upper', aspect='auto',
          extent=[0, FRAME_W, FRAME_H, 0], cmap='hot', interpolation='gaussian')
ax.set_title('All-Player Position Heatmap', fontsize=11)
ax.set_xlabel('X (px)'); ax.set_ylabel('Y (px)')

# Confidence CDF
ax = fig.add_subplot(gs[2, 2])
for cls_id, lbl in CLASS_LABELS.items():
    sub = tracking[tracking['class_id'] == cls_id]['confidence'].sort_values()
    if sub.empty: continue
    cdf = np.arange(1, len(sub) + 1) / len(sub)
    ax.plot(sub.values, cdf, lw=1.8, color=CLASS_COLORS.get(cls_id, 'grey'), label=lbl)
ax.axvline(0.5, color='red', ls='--', lw=1.2, label='thr=0.5')
ax.set_title('Confidence CDF', fontsize=11)
ax.set_xlabel('Confidence'); ax.set_ylabel('CDF'); ax.legend(fontsize=7)

# Speed KDE
ax = fig.add_subplot(gs[2, 3])
for team_id in [0, 1]:
    sub = kin_df[kin_df['team_id'] == team_id]['speed_km_h_clean']
    if sub.empty: continue
    sub.plot.kde(ax=ax, color=TEAM_COLORS[team_id], lw=2, label=TEAM_LABELS[team_id])
ax.set_title('Speed KDE', fontsize=11)
ax.set_xlabel('Speed (km/h)'); ax.set_ylabel('Density'); ax.set_xlim(left=0); ax.legend(fontsize=8)

plt.savefig('data/annotations/summary_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → data/annotations/summary_dashboard.png')


---
## 10 · Printable Stats Report

In [ ]:
LINE = '═' * 55
print(LINE)
print('  FOOTBALL TRACKING — STATS REPORT')
print(LINE)

print('\n VIDEO & DETECTIONS')
print(f'  Frames with detections : {n_frames:,}')
print(f'  Total detection rows   : {len(tracking):,}')
print(f'  Mean confidence        : {conf_mean:.3f}')
print(f'  Low-conf rows (<0.5)   : {conf_low:,}  ({100*conf_low/len(tracking):.1f}%)')
print(f'  Avg detections/frame   : {len(tracking)/n_frames:.1f}')

print('\n BALL')
print(f'  Frames with ball info  : {total_frames_with_ball:,}')
print(f'  Interpolated frames    : {int(interp_frames):,}  ({100*interp_frames/max(1,total_frames_with_ball):.1f}%)')

print('\n TEAM POSSESSION')
for _, row in poss_sum.sort_values('team_id').iterrows():
    label = TEAM_LABELS.get(int(row['team_id']), str(int(row['team_id'])))
    print(f'  {label:<12} : {row["possession_pct"]:5.1f}%')

print('\n PLAYER SPEEDS')
ps_players = player_sum[player_sum['class_id'].isin([1, 2])]
for team_id in [0, 1]:
    sub = ps_players[ps_players['team_id'] == team_id]
    if sub.empty: continue
    print(f'  {TEAM_LABELS[team_id]}')
    print(f'    Top speed : {sub["top_speed_km_h"].max():.1f} km/h  (ID {sub.loc[sub["top_speed_km_h"].idxmax(), "object_id"]})')
    print(f'    Mean avg  : {sub["avg_speed_km_h"].mean():.1f} km/h')
    print(f'    Players   : {len(sub)}')

print('\n TRACKING')
print(f'  Total tracked objects  : {n_objects}')
print(f'  Median track length    : {players_df["track_len"].median():.0f} frames')
print(f'  Short tracks (<10 f)   : {(players_df["track_len"] < 10).sum()}')

print(f'\n{LINE}')
print('  Plots saved to data/annotations/')
print(LINE)


---
## 11 · Replay Detection & Data Cleaning

Broadcast football video regularly cuts to slow-motion replays, then back to
live play.  The tracking pipeline has **no awareness** of these cuts, which
corrupts the data in four ways:

| Signal | What happens during a replay |
|--------|------------------------------|
| **Speed (km/h)** | When live play resumes the tracker sees a player teleport from their replay position back to the pitch → enormous fake speed spike |
| **Object IDs** | All ByteTrack IDs expire during the replay; every player gets a new ID when live play resumes → `n_objects` is hugely inflated |
| **Ball position** | The interpolator keeps extrapolating using pre-replay velocity → fake ball coordinates in the CSV |
| **Possession %** | Possession is awarded against fake ball positions → stats are wrong |

Because we are working with the already-produced CSV (not the raw video), we
detect replay-contaminated frames using **three data-only signals**:

1. **Speed spikes** — rows where `speed_km_h` is > mean + 4σ (physically
   impossible for a real player movement at the true frame rate).
2. **Position jumps** — for each track, frames where the feet position jumps
   more than a threshold between consecutive detections.
3. **ID bursts** — frames where many new `object_id` values appear together
   in a short window (replay boundary causes a flood of new IDs).

In [ ]:
# ── Signal 1: Speed spikes ────────────────────────────────────────────────────
# Use the RAW speed (not capped) so we can see the full extent of the artefacts.
players_raw = enriched[enriched['class_id'].isin([1, 2])].copy()

spd_mean = players_raw['speed_km_h'].mean()
spd_std  = players_raw['speed_km_h'].std()
SPIKE_THRESH = spd_mean + 4 * spd_std      # 4-sigma: physically impossible

spike_frames = set(players_raw.loc[players_raw['speed_km_h'] > SPIKE_THRESH, 'frame_id'])

print(f'Speed distribution (players): mean={spd_mean:.1f}  std={spd_std:.1f}  km/h')
print(f'Spike threshold (mean+4σ)   : {SPIKE_THRESH:.1f} km/h')
print(f'Spike rows                  : {(players_raw["speed_km_h"] > SPIKE_THRESH).sum():,}')
print(f'Frames with speed spikes    : {len(spike_frames):,}')

# ── Signal 2: Position jumps ──────────────────────────────────────────────────
# For each track, compute Euclidean distance between consecutive frame positions.
JUMP_THRESH_PX = 120    # > 120 px between consecutive frames = impossible real motion

jump_frames = set()
for obj_id, grp in enriched[enriched['class_id'].isin([1, 2])].groupby('object_id'):
    grp_s = grp.sort_values('frame_id')
    dx = grp_s['feet_x'].diff()
    dy = grp_s['feet_y'].diff()
    df = grp_s['frame_id'].diff().replace(0, 1)      # avoid div-by-zero
    dist_per_frame = np.hypot(dx, dy) / df            # px per frame
    mask = dist_per_frame > JUMP_THRESH_PX
    jump_frames.update(grp_s.loc[mask, 'frame_id'].tolist())

print(f'\nJump threshold              : {JUMP_THRESH_PX} px/frame')
print(f'Frames with position jumps  : {len(jump_frames):,}')

# ── Signal 3: ID bursts ───────────────────────────────────────────────────────
# Find the first frame each object_id appears in, then count how many new IDs
# appear within a rolling window of BURST_WINDOW frames.
BURST_WINDOW = 15    # frames
BURST_THRESH = 6     # new IDs in that window = replay boundary

first_seen = (
    enriched[enriched['class_id'].isin([1, 2])]
    .groupby('object_id')['frame_id']
    .min()
    .reset_index()
    .rename(columns={'frame_id': 'first_frame'})
)
first_seen_counts = first_seen.groupby('first_frame').size().rename('new_ids')
first_seen_counts = first_seen_counts.reindex(
    range(enriched['frame_id'].min(), enriched['frame_id'].max() + 1), fill_value=0
)
rolling_new_ids = first_seen_counts.rolling(BURST_WINDOW, min_periods=1).sum()
burst_frames = set(rolling_new_ids[rolling_new_ids >= BURST_THRESH].index.tolist())

print(f'\nID-burst window             : {BURST_WINDOW} frames')
print(f'ID-burst threshold          : {BURST_THRESH} new IDs')
print(f'Frames with ID bursts       : {len(burst_frames):,}')

# ── Combine signals → suspect replay frames ───────────────────────────────────
# A frame is flagged if it triggers ANY signal.
# Then we grow each flagged frame by ±HALO frames to capture the full replay
# region (replays typically last several seconds).
HALO = 20   # frames to expand around each flagged frame

raw_suspect = spike_frames | jump_frames | burst_frames
replay_frames = set()
for f in raw_suspect:
    replay_frames.update(range(f - HALO, f + HALO + 1))

# Clamp to valid frame range
fmin, fmax = int(enriched['frame_id'].min()), int(enriched['frame_id'].max())
replay_frames = {f for f in replay_frames if fmin <= f <= fmax}

print(f'\n──────────────────────────────────────')
print(f'Raw suspect frames          : {len(raw_suspect):,}')
print(f'After ±{HALO}-frame expansion : {len(replay_frames):,}')
print(f'Estimated replay fraction   : {100*len(replay_frames)/(fmax-fmin+1):.1f}% of all frames')

In [ ]:
# ── Visualise the three signals and the detected replay regions ───────────────
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('Replay Artifact Detection — Three Data Signals', fontsize=14, fontweight='bold')

all_frames = np.arange(fmin, fmax + 1)

# Shade detected replay regions on every panel
def shade_replays(ax):
    in_block = False
    start = None
    sorted_rf = sorted(replay_frames)
    for i, f in enumerate(sorted_rf):
        if not in_block:
            start = f
            in_block = True
        if i == len(sorted_rf) - 1 or sorted_rf[i + 1] > f + 1:
            ax.axvspan(start, f, color='#FF5722', alpha=0.15, label='_nolegend_')
            in_block = False
    # Add one legend entry
    ax.axvspan(0, 0, color='#FF5722', alpha=0.35, label='Detected replay region')

# ── Panel 1: Raw speed per frame (player rows) ───────────────────────────────
ax = axes[0]
spd_by_frame = players_raw.groupby('frame_id')['speed_km_h'].max()
ax.plot(spd_by_frame.index, spd_by_frame.values, lw=0.6, color='steelblue', alpha=0.7)
ax.axhline(SPIKE_THRESH, color='red', ls='--', lw=1.5, label=f'spike threshold ({SPIKE_THRESH:.0f} km/h)')
shade_replays(ax)
ax.set_ylabel('Max speed (km/h)')
ax.set_title('Signal 1 — Speed Spikes')
ax.legend(fontsize=8)

# ── Panel 2: Max position jump per frame ─────────────────────────────────────
ax = axes[1]
jump_by_frame = {}
for obj_id, grp in enriched[enriched['class_id'].isin([1, 2])].groupby('object_id'):
    grp_s = grp.sort_values('frame_id')
    dx = grp_s['feet_x'].diff()
    dy = grp_s['feet_y'].diff()
    df_gap = grp_s['frame_id'].diff().replace(0, 1)
    dpf = np.hypot(dx, dy) / df_gap
    for fid, val in zip(grp_s['frame_id'], dpf):
        if not np.isnan(val):
            jump_by_frame[fid] = max(jump_by_frame.get(fid, 0), val)

jump_series = pd.Series(jump_by_frame).reindex(all_frames, fill_value=0)
ax.plot(jump_series.index, jump_series.values, lw=0.6, color='purple', alpha=0.7)
ax.axhline(JUMP_THRESH_PX, color='red', ls='--', lw=1.5, label=f'jump threshold ({JUMP_THRESH_PX} px/frame)')
shade_replays(ax)
ax.set_ylabel('Max jump (px/frame)')
ax.set_title('Signal 2 — Position Jumps')
ax.legend(fontsize=8)

# ── Panel 3: Rolling new ID count ────────────────────────────────────────────
ax = axes[2]
rolling_plot = rolling_new_ids.reindex(all_frames, fill_value=0)
ax.fill_between(rolling_plot.index, rolling_plot.values, color='orange', alpha=0.6, lw=0)
ax.axhline(BURST_THRESH, color='red', ls='--', lw=1.5, label=f'burst threshold ({BURST_THRESH} new IDs/{BURST_WINDOW}f)')
shade_replays(ax)
ax.set_ylabel(f'New IDs / {BURST_WINDOW} frames')
ax.set_title('Signal 3 — ID Bursts')
ax.legend(fontsize=8)

# ── Panel 4: Combined replay mask ────────────────────────────────────────────
ax = axes[3]
replay_mask = pd.Series(
    [1 if f in replay_frames else 0 for f in all_frames],
    index=all_frames
)
ax.fill_between(all_frames, replay_mask.values, color='#FF5722', alpha=0.6, lw=0, label='Replay / contaminated')
ax.fill_between(all_frames, 1 - replay_mask.values, color='#4CAF50', alpha=0.35, lw=0, label='Live play')
ax.set_ylim(0, 1.1)
ax.set_yticks([])
ax.set_title('Combined — Live vs Replay Regions')
ax.set_xlabel('Frame')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('data/annotations/replay_detection.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/annotations/replay_detection.png')

---
## 12 · Before vs After — Effect of Removing Replay Frames

In [ ]:
# ── Build clean (replay-filtered) dataframes ─────────────────────────────────
df_clean    = df[~df['frame_id'].isin(replay_frames)].copy()
enr_clean   = enriched[~enriched['frame_id'].isin(replay_frames)].copy()
track_clean = tracking[~tracking['frame_id'].isin(replay_frames)].copy()

# Recompute per-object summary on clean data ──────────────────────────────────
from collections import defaultdict

def quick_summary(df_in):
    """Minimal player summary from a tracking dataframe."""
    rows = []
    for obj_id, grp in df_in[df_in['class_id'].isin([1, 2])].groupby('object_id'):
        speeds = grp['speed_km_h'].values
        rows.append({
            'object_id':      obj_id,
            'team_id':        grp['team_id'].iloc[0],
            'total_frames':   len(grp),
            'top_speed_km_h': float(speeds.max()),
            'avg_speed_km_h': float(speeds.mean()),
        })
    return pd.DataFrame(rows)

ps_raw   = quick_summary(enriched)
ps_clean = quick_summary(enr_clean)

# Possession percentages before & after ──────────────────────────────────────
def possession_pct(df_in):
    poss = df_in[df_in['has_possession'] == 1][['frame_id', 'team_id']].drop_duplicates('frame_id')
    if poss.empty:
        return {}
    total = len(poss)
    return {t: round(100 * n / total, 1) for t, n in poss['team_id'].value_counts().items()}

poss_before = possession_pct(enriched)
poss_after  = possession_pct(enr_clean)

print('──────── BEFORE (raw) ────────────────────────────────────────────')
print(f'  Tracked objects (players)  : {ps_raw["object_id"].nunique()}')
print(f'  Max speed ever seen        : {ps_raw["top_speed_km_h"].max():.1f} km/h')
print(f'  Mean of top speeds         : {ps_raw["top_speed_km_h"].mean():.1f} km/h')
print(f'  Possession: {poss_before}')

print()
print('──────── AFTER  (replay-cleaned) ────────────────────────────────')
print(f'  Tracked objects (players)  : {ps_clean["object_id"].nunique()}')
print(f'  Max speed ever seen        : {ps_clean["top_speed_km_h"].max():.1f} km/h')
print(f'  Mean of top speeds         : {ps_clean["top_speed_km_h"].mean():.1f} km/h')
print(f'  Possession: {poss_after}')

print()
print(f'  Rows removed               : {len(enriched) - len(enr_clean):,}  ({100*(len(enriched)-len(enr_clean))/len(enriched):.1f}%)')
print(f'  Objects removed (ghosts)   : {ps_raw["object_id"].nunique() - ps_clean["object_id"].nunique()}')

In [ ]:
# ── Visual comparison: 6 panels before vs after ──────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Before vs After Replay Cleaning', fontsize=15, fontweight='bold')

BINS = 60

# ── Row 0: Speed distribution ─────────────────────────────────────────────────
for col, (label, ps, color) in enumerate([
    ('Before (raw)',     ps_raw,   '#EF5350'),
    ('After (cleaned)',  ps_clean, '#42A5F5'),
]):
    ax = axes[0, col]
    data = ps['top_speed_km_h'].clip(upper=150)
    ax.hist(data, bins=BINS, color=color, edgecolor='white', lw=0.4, density=True)
    ax.axvline(data.median(), color='black', ls='--', lw=1.5, label=f'median={data.median():.0f}')
    ax.set_title(f'Top Speed Distribution\n{label}')
    ax.set_xlabel('Top speed (km/h)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

# ── Row 1: Track-length distribution ─────────────────────────────────────────
tlen_raw   = tracking[tracking['class_id'].isin([1, 2])].groupby('object_id').size()
tlen_clean = track_clean[track_clean['class_id'].isin([1, 2])].groupby('object_id').size()

for col, (label, tlen, color) in enumerate([
    ('Before (raw)',     tlen_raw,   '#EF5350'),
    ('After (cleaned)',  tlen_clean, '#42A5F5'),
]):
    ax = axes[1, col]
    ax.hist(tlen, bins=40, color=color, edgecolor='white', lw=0.4)
    ax.axvline(tlen.median(), color='black', ls='--', lw=1.5, label=f'median={tlen.median():.0f}')
    short = (tlen < 10).sum()
    ax.set_title(f'Track Length Distribution\n{label}  (short<10: {short})')
    ax.set_xlabel('Track length (frames)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

# ── Row 2: Possession bar ─────────────────────────────────────────────────────
for col, (label, poss, color) in enumerate([
    ('Before (raw)',     poss_before, '#EF5350'),
    ('After (cleaned)',  poss_after,  '#42A5F5'),
]):
    ax = axes[2, col]
    if not poss:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue
    bar_labels  = [TEAM_LABELS.get(t, str(t)) for t in sorted(poss)]
    bar_values  = [poss[t] for t in sorted(poss)]
    bar_colors  = [TEAM_COLORS.get(t, 'grey') for t in sorted(poss)]
    bars = ax.bar(bar_labels, bar_values, color=bar_colors, edgecolor='white')
    for bar, val in zip(bars, bar_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(bar_values) * 1.2)
    ax.set_title(f'Possession %\n{label}')
    ax.set_ylabel('Possession %')

plt.tight_layout()
plt.savefig('data/annotations/replay_before_after.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/annotations/replay_before_after.png')

In [ ]:
# ── Export the clean datasets ─────────────────────────────────────────────────
out_dir = ANNOT_DIR

enr_clean.to_csv(out_dir / 'tracking_enriched_clean.csv', index=False)
track_clean.to_csv(out_dir / 'tracking_clean.csv', index=False)

print('Clean datasets saved:')
for name in ['tracking_clean.csv', 'tracking_enriched_clean.csv']:
    p = out_dir / name
    print(f'  {name:<40} {p.stat().st_size/1024:.1f} KB  ({len(pd.read_csv(p)):,} rows)')

print()
print('Use these files for any downstream analysis instead of the originals.')
print('The replay-contaminated frames have been removed from both.')